# M06 — Rolling VaR Backtesting and ES Tail Diagnostics

This notebook reproduces the M06 no-lookahead backtest in a fresh Google Colab runtime. It reports VaR coverage and independence tests separately from Expected Shortfall tail-severity diagnostics. Statistical rejection is model evidence, not a pipeline failure.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JoyWu-302121/market_risk.git'
PROJECT_DIR = Path('/content/market_risk') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB and not (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
elif IN_COLAB:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print(f'Project ready at {PROJECT_DIR}')

## Run the accepted M06 pipeline

The reusable runner downloads the official GSW data, constructs rolling forecasts, preserves failed outcomes and warm-up dates, and writes forecast-level and summary artifacts.

In [ ]:
import json

OUTPUT_ROOT = PROJECT_DIR / 'data'
completed = subprocess.run(
    [sys.executable, 'scripts/run_m06_backtesting.py', '--output-root', str(OUTPUT_ROOT)],
    check=True,
    capture_output=True,
    text=True,
)
report = json.loads(completed.stdout)
assert report['status'] == 'PASS', report['checks']
print(f"M06 status: {report['status']}")
print(f"Backtest end date: {report['portfolio']['valuation_date']}")
print(report['forecast_audit']['status_counts'])

## Review VaR coverage and independence

A p-value below 5% rejects the corresponding null at the configured significance level. The primary model uses the 750-observation window.

In [ ]:
import pandas as pd
from IPython.display import display

var_summary = pd.read_csv(report['artifacts']['var_summary_path'])
display(
    var_summary[[
        'window_size', 'confidence_level', 'kupiec_observation_count',
        'kupiec_exception_count', 'kupiec_expected_exception_count',
        'kupiec_exception_rate', 'kupiec_p_value', 'independence_p_value',
        'conditional_coverage_p_value', 'maximum_consecutive_exceptions'
    ]].style.format({
        'confidence_level': '{:.1%}',
        'kupiec_expected_exception_count': '{:,.1f}',
        'kupiec_exception_rate': '{:.2%}',
        'kupiec_p_value': '{:.4f}',
        'independence_p_value': '{:.4f}',
        'conditional_coverage_p_value': '{:.4f}',
    })
)

In [ ]:
import matplotlib.pyplot as plt

forecasts = pd.read_csv(report['artifacts']['forecasts_path'], parse_dates=['forecast_date', 'realization_date'])
primary_99 = forecasts.loc[
    (forecasts['window_size'] == 750)
    & (forecasts['measure'] == 'VaR')
    & (forecasts['confidence_level'] == 0.99)
].copy()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(primary_99['realization_date'], primary_99['realized_loss'] / 1_000, linewidth=0.8, label='Realized loss')
ax.plot(primary_99['realization_date'], primary_99['forecast_value'] / 1_000, linewidth=1.0, label='99% VaR')
exceptions = primary_99.loc[primary_99['exception']]
ax.scatter(exceptions['realization_date'], exceptions['realized_loss'] / 1_000, color='red', s=16, label='Exception', zorder=3)
ax.set(title='750-Observation 99% Historical VaR Backtest', ylabel='USD thousands', xlabel='Realization date')
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Review ES tail severity separately

Tail dates are selected with empirical VaR at the same confidence level as ES. The ratio compares mean realized loss with mean forecast ES on those dates.

In [ ]:
es_summary = pd.read_csv(report['artifacts']['es_summary_path'])
display(
    es_summary[[
        'window_size', 'confidence_level', 'observation_count',
        'tail_observation_count', 'expected_tail_count',
        'mean_realized_tail_loss', 'mean_forecast_es_on_tail_dates',
        'tail_loss_to_es_ratio', 'es_breach_count'
    ]].style.format({
        'confidence_level': '{:.1%}',
        'expected_tail_count': '{:,.2f}',
        'mean_realized_tail_loss': '${:,.2f}',
        'mean_forecast_es_on_tail_dates': '${:,.2f}',
        'tail_loss_to_es_ratio': '{:.4f}',
    })
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = [f"{int(row.window_size)} / {row.confidence_level:.1%}" for row in es_summary.itertuples()]
colors = ['#b24745' if value > 1 else '#3c78a8' for value in es_summary['tail_loss_to_es_ratio']]
ax.bar(labels, es_summary['tail_loss_to_es_ratio'], color=colors)
ax.axhline(1.0, color='black', linewidth=1)
ax.set(title='Realized Tail Loss / Forecast ES', xlabel='Window / ES level', ylabel='Ratio')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Inspect audit statuses and verify completion

In [ ]:
audit = pd.read_csv(report['artifacts']['audit_path'])
display(audit['status'].value_counts().rename('row_count').to_frame())
assert all(report['checks'].values())
assert len(var_summary) == 6
assert len(es_summary) == 6
assert (audit['status'] == 'missing_realized_input').any()
assert (audit['status'] == 'insufficient_history').any()
print('M06 backtesting implementation checks: PASS')

M06 validates the historical-simulation model only. M07 will add parametric normal and PCA Monte Carlo benchmark models, which require separate validation.